In [5]:
# ============================================
# DATA MINING LAB: FEATURE SELECTION
# 1. Information Gain (Entropy)
# 2. Pearson Correlation
# 3. ANOVA F-Test
# ============================================

import pandas as pd
import math

# Load dataset
df = pd.read_csv("477-lab-exam.csv")

print("Dataset:")
print(df.head())

# Separate features and target
X = df.drop(columns=["Record_ID", "Class"], errors="ignore")
Y = df["Class"]

features = list(X.columns)
n = len(Y)



Dataset:
   Record_ID  Feature_1  Feature_2  Feature_3  Feature_4  Feature_5  \
0          1      47.52     496.00      44.74      51.11      51.41   
1          2      37.90      42.16      42.16      41.12     426.00   
2          3      61.16      55.14      57.20      62.94      59.20   
3          4      55.57      54.20      46.56      50.81      51.37   
4          5      58.94      60.95      59.43     591.00      56.81   

   Feature_6  Feature_7  Feature_8  Feature_9  ...  Feature_12  Feature_13  \
0      49.93      49.83      45.52      50.95  ...       51.49       56.14   
1      42.74      44.62      44.79     519.00  ...       44.26       61.28   
2      51.24      45.55      57.14      50.96  ...       49.51       56.43   
3      51.18      50.21      56.32      46.84  ...       41.49       48.29   
4      51.92     527.00      56.32      50.46  ...       56.24       51.95   

   Feature_14  Feature_15  Feature_16  Feature_17  Feature_18  Feature_19  \
0       51.67     

In [6]:
# ============================================
# PART 1: INFORMATION GAIN (ENTROPY)
# ============================================

# Calculate entropy manually

def entropy(data):
    classes = list(set(data))
    total = len(data)
    ent = 0
    for c in classes:
        count = 0
        for value in data:
            if value == c:
                count += 1
        p = count / total

        if p > 0:
            ent = ent - p * math.log2(p)

    return ent


# Discretize continuous features into 5 equal-width bins

def discretize(data):

    minimum = min(data)
    maximum = max(data)
    bins = 5
    width = (maximum - minimum) / bins
    result = []

    for value in data:
        if width == 0:
            result.append(0)
        else:
            b = int((value - minimum) / width)

            if b >= bins:
                b = bins - 1

            result.append(b)

    return result


# Calculate Information Gain

parent_entropy = entropy(Y)
ig_scores = {}

for feature in features:
    values = discretize(X[feature])
    unique_values = list(set(values))
    weighted_entropy = 0
    for v in unique_values:
        subset = []
        for i in range(n):
            if values[i] == v:
                subset.append(Y.iloc[i])

        weight = len(subset) / n
        weighted_entropy += weight * entropy(subset)

    ig = parent_entropy - weighted_entropy
    ig_scores[feature] = ig


# Rank Information Gain

ig_rank = sorted(ig_scores.items(),
                 key=lambda x: x[1],
                 reverse=True)

print("\n================================")
print("INFORMATION GAIN RESULTS")
print("================================")

for feature, score in ig_rank:
    print(feature, ":", round(score, 5))

top_ig = [feature for feature, score in ig_rank[:10]]

print("\nTop 10 Features (Information Gain):")
print(top_ig)




INFORMATION GAIN RESULTS
Feature_8 : 0.70297
Feature_6 : 0.2788
Feature_2 : 0.19683
Feature_14 : 0.19521
Feature_1 : 0.18352
Feature_19 : 0.17812
Feature_15 : 0.17002
Feature_17 : 0.16067
Feature_11 : 0.15702
Feature_4 : 0.15683
Feature_5 : 0.1502
Feature_18 : 0.14632
Feature_20 : 0.14193
Feature_9 : 0.11997
Feature_10 : 0.11862
Feature_13 : 0.09791
Feature_3 : 0.08926
Feature_16 : 0.08926
Feature_12 : 0.0702
Feature_7 : 0.04605

Top 10 Features (Information Gain):
['Feature_8', 'Feature_6', 'Feature_2', 'Feature_14', 'Feature_1', 'Feature_19', 'Feature_15', 'Feature_17', 'Feature_11', 'Feature_4']


In [7]:
# ============================================
# PART 2: PEARSON CORRELATION
# ============================================

def pearson(x, y):
    n = len(x)
    mean_x = sum(x) / n
    mean_y = sum(y) / n
    numerator = 0
    sum_x = 0
    sum_y = 0

    for i in range(n):
        dx = x[i] - mean_x
        dy = y[i] - mean_y
        numerator += dx * dy
        sum_x += dx ** 2
        sum_y += dy ** 2

    denominator = math.sqrt(sum_x * sum_y)

    if denominator == 0:
        return 0

    return numerator / denominator


pearson_scores = {}

for feature in features:
    x = list(X[feature])
    y = list(Y)
    r = pearson(x, y)
    pearson_scores[feature] = abs(r)


# Rank Pearson Correlation

pearson_rank = sorted(pearson_scores.items(),
                      key=lambda x: x[1],
                      reverse=True)

print("\n================================")
print("PEARSON CORRELATION RESULTS")
print("================================")

for feature, score in pearson_rank:
    print(feature, ":", round(score, 5))

top_pearson = [feature for feature, score in pearson_rank[:10]]

print("\nTop 10 Features (Pearson):")
print(top_pearson)



PEARSON CORRELATION RESULTS
Feature_8 : 0.762
Feature_20 : 0.3402
Feature_11 : 0.33208
Feature_13 : 0.29855
Feature_1 : 0.24037
Feature_4 : 0.19577
Feature_15 : 0.19485
Feature_16 : 0.17072
Feature_6 : 0.16646
Feature_7 : 0.15286
Feature_12 : 0.14472
Feature_3 : 0.12951
Feature_19 : 0.11965
Feature_2 : 0.08538
Feature_5 : 0.08255
Feature_17 : 0.07319
Feature_10 : 0.04312
Feature_14 : 0.03346
Feature_9 : 0.00845
Feature_18 : 0.00515

Top 10 Features (Pearson):
['Feature_8', 'Feature_20', 'Feature_11', 'Feature_13', 'Feature_1', 'Feature_4', 'Feature_15', 'Feature_16', 'Feature_6', 'Feature_7']


In [8]:
# ============================================
# PART 3: ANOVA F-TEST
# ============================================

def anova_f(x, y):
    classes = list(set(y))
    total = len(x)
    grand_mean = sum(x) / total
    between = 0
    within = 0

    for c in classes:
        group = []
        for i in range(total):
            if y[i] == c:
                group.append(x[i])

        group_mean = sum(group) / len(group)

        # Between-group sum of squares
        between += len(group) * (group_mean - grand_mean) ** 2

        # Within-group sum of squares
        for value in group:
            within += (value - group_mean) ** 2

    k = len(classes)
    df_between = k - 1
    df_within = total - k
    ms_between = between / df_between
    ms_within = within / df_within

    if ms_within == 0:
        return 0

    return ms_between / ms_within


anova_scores = {}

for feature in features:
    x = list(X[feature])
    y = list(Y)
    f = anova_f(x, y)
    anova_scores[feature] = f

# Rank ANOVA

anova_rank = sorted(anova_scores.items(),
                    key=lambda x: x[1],
                    reverse=True)

print("\n================================")
print("ANOVA F-TEST RESULTS")
print("================================")

for feature, score in anova_rank:
    print(feature, ":", round(score, 5))

top_anova = [feature for feature, score in anova_rank[:10]]

print("\nTop 10 Features (ANOVA):")
print(top_anova)



ANOVA F-TEST RESULTS
Feature_8 : 17.17696
Feature_17 : 3.56703
Feature_11 : 2.7348
Feature_13 : 2.23213
Feature_20 : 2.09317
Feature_3 : 1.6782
Feature_15 : 1.51501
Feature_16 : 1.51002
Feature_6 : 1.45245
Feature_4 : 1.40227
Feature_1 : 1.39004
Feature_10 : 1.05836
Feature_2 : 1.03136
Feature_12 : 0.97378
Feature_18 : 0.87447
Feature_14 : 0.72283
Feature_5 : 0.71809
Feature_19 : 0.61981
Feature_7 : 0.61411
Feature_9 : 0.06931

Top 10 Features (ANOVA):
['Feature_8', 'Feature_17', 'Feature_11', 'Feature_13', 'Feature_20', 'Feature_3', 'Feature_15', 'Feature_16', 'Feature_6', 'Feature_4']


In [9]:
# ============================================
# PART 4: COMPARISON OF METHODS
# ============================================

print("\n================================")
print("COMPARISON OF FEATURE SELECTION")
print("================================")

# Common features between Information Gain and Pearson

common = []

for feature in top_ig:
    if feature in top_pearson:
        common.append(feature)

print("\nCommon: Information Gain & Pearson")
print(common)


# Features selected only by Information Gain

only_ig = []

for feature in top_ig:
    if feature not in top_pearson:
        only_ig.append(feature)

print("\nOnly Information Gain:")
print(only_ig)


# Features selected only by Pearson

only_pearson = []

for feature in top_pearson:
    if feature not in top_ig:
        only_pearson.append(feature)

print("\nOnly Pearson:")
print(only_pearson)


# Compare all three methods

common_three = []

for feature in top_ig:
    if feature in top_pearson and feature in top_anova:
        common_three.append(feature)

print("\nCommon in ALL THREE methods:")
print(common_three)
print("\n================================")
print("FINAL SUMMARY")
print("================================")
print("Information Gain Top 10:", top_ig)
print("Pearson Top 10:", top_pearson)
print("ANOVA Top 10:", top_anova)
print("\nAnalysis completed successfully!")


COMPARISON OF FEATURE SELECTION

Common: Information Gain & Pearson
['Feature_8', 'Feature_6', 'Feature_1', 'Feature_15', 'Feature_11', 'Feature_4']

Only Information Gain:
['Feature_2', 'Feature_14', 'Feature_19', 'Feature_17']

Only Pearson:
['Feature_20', 'Feature_13', 'Feature_16', 'Feature_7']

Common in ALL THREE methods:
['Feature_8', 'Feature_6', 'Feature_15', 'Feature_11', 'Feature_4']

FINAL SUMMARY
Information Gain Top 10: ['Feature_8', 'Feature_6', 'Feature_2', 'Feature_14', 'Feature_1', 'Feature_19', 'Feature_15', 'Feature_17', 'Feature_11', 'Feature_4']
Pearson Top 10: ['Feature_8', 'Feature_20', 'Feature_11', 'Feature_13', 'Feature_1', 'Feature_4', 'Feature_15', 'Feature_16', 'Feature_6', 'Feature_7']
ANOVA Top 10: ['Feature_8', 'Feature_17', 'Feature_11', 'Feature_13', 'Feature_20', 'Feature_3', 'Feature_15', 'Feature_16', 'Feature_6', 'Feature_4']

Analysis completed successfully!
